# ELECTRO-VOLT — 02. Preprocesamiento y Decisión de Clases Físicas
**Proyecto Integrador · Maestría en IA · UEES**

**Autores:** Jordan Vera Coello & Jorge Estrella Frere

En este notebook se aborda la decisión técnica y física central del proyecto:
1. Demostración experimental de **indistinguibilidad física** entre fallas trifásicas `LLL` y `LLLG` usando las 6 señales de fase.
2. Justificación física y normativa para la docente evaluadora.
3. Fusión oficial a **5 clases físicas** (`Normal`, `LG`, `LL`, `LLG`, `Trifásica`).
4. Implementación del enfoque alternativo documental: Feature Engineering de 13 variables ($I_0, V_0$).
5. Partición estratificada con mezcla (80% entrenamiento / 20% prueba).
6. Exportación del conjunto de prueba representativo `data/processed/ejemplos_prueba.csv`.


In [ ]:
import sys
from pathlib import Path

raiz = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(raiz) not in sys.path:
    sys.path.append(str(raiz))

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

from src.data_processing import (
    cargar_datos,
    mapear_clases,
    agregar_feature_engineering,
    dividir_datos,
    exportar_ejemplos_prueba,
)
from src.utils import COLUMNAS, COLUMNAS_FE_13, ORDEN_CLASES, SEMILLA

sns.set_theme(style="whitegrid")
print("[OK] Módulos de preprocesamiento listos.")


## 1. Demostración experimental de indistinguibilidad
Entrenamos un clasificador Random Forest únicamente sobre el subconjunto de fallas trifásicas (`LLL` y `LLLG`) usando exclusivamente las 6 señales de fase (`Va, Vb, Vc, Ia, Ib, Ic`).


In [ ]:
df_crudo = cargar_datos()
df_6 = mapear_clases(df_crudo, modo="6_clases")

# Filtrar únicamente fallas LLL y LLLG
sub = df_6[df_6["clase_6"].isin(["LLL", "LLLG"])]
X_sub = sub[COLUMNAS]
y_sub = sub["clase_6"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X_sub, y_sub, test_size=0.20, stratify=y_sub, shuffle=True, random_state=SEMILLA
)

rf_exp = RandomForestClassifier(n_estimators=100, random_state=SEMILLA, class_weight="balanced")
rf_exp.fit(X_tr, y_tr)
acc_exp = accuracy_score(y_te, rf_exp.predict(X_te))
azar = y_te.value_counts(normalize=True).max()

print(f"Accuracy del modelo separando LLL vs LLLG: {acc_exp:.4f}")
print(f"Accuracy de una predicción ingenua por azar:   {azar:.4f}")
print("CONCLUSIÓN EXPERIMENTAL: El modelo no logra distinguir LLL de LLLG con las señales de fase; su precisión es equivalente al azar.")


## 2. Justificación Física del Ajuste a 5 clases
> ### Fundamento Eléctrico
> - **Simetría y suma fasorial:** En una falla trifásica balanceada, las corrientes de cortocircuito tienen idéntica magnitud y un desfase exacto de 120° ($I_a + I_b + I_c = 0$). Por ende, la corriente por el neutro o la conexión a tierra es matemáticamente nula ($I_N = 0$).
> - **Comportamiento en red:** La presencia o ausencia de tierra física no altera las tensiones ni corrientes de fase medidas por los transformadores de potencial (TP) y de corriente (TC).
> - **Operación de subestación:** La protección de fase despeja la falla trifásica disparando las tres fases simultáneamente en ambos casos, sin requerir maniobras operativas distintas.
> - **Artefacto de simulación descartado:** Aunque la suma $Ia+Ib+Ic$ en el dataset sintético arroja un residuo de ~2 A en LLL vs 0 A en LLLG, esto corresponde a un artefacto numérico de cálculo en MATLAB/Simulink y no a una física real de campo.


## 3. Fusión oficial a 5 clases (`Trifásica`)
Aplicamos la unificación de `LLL` y `LLLG` en la etiqueta unificada `Trifásica`.


In [ ]:
df_5 = mapear_clases(df_crudo, modo="5_clases")
conteo_5 = df_5["clase"].value_counts().reindex(ORDEN_CLASES)
print("Distribución de las 5 clases oficiales:\n", conteo_5)

plt.figure(figsize=(8, 4))
conteo_5.plot(kind="bar", color="seagreen")
plt.title("Muestras por clase tras fusión física (5 Clases)", fontsize=13)
plt.ylabel("Muestras")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 4. Enfoque alternativo documental
Calculamos las 7 variables residuales ($I_0, V_0, |I_0|, |V_0|$, extremos) utilizadas en el reporte de diagnóstico para el caso de 6 clases.


In [ ]:
df_fe = agregar_feature_engineering(df_crudo)
print("13 Variables generadas:", COLUMNAS_FE_13)
df_fe[COLUMNAS_FE_13].head(3)


## 5. Partición estratificada 80/20 y exportación de ejemplos
Dividimos los datos para modelado y generamos `data/processed/ejemplos_prueba.csv`.


In [ ]:
X_ent, X_pru, y_ent, y_pru, idx_ent, idx_pru = dividir_datos(
    df_5, columnas=COLUMNAS, columna_objetivo="clase", test_size=0.20, random_state=SEMILLA
)
print(f"Muestras de entrenamiento: {len(X_ent)} (80%)")
print(f"Muestras de prueba:        {len(X_pru)} (20%)")

ejemplos = exportar_ejemplos_prueba(df_5, idx_pru, n_por_clase=3)
print(f"[OK] Exportados {len(ejemplos)} ejemplos a data/processed/ejemplos_prueba.csv")
ejemplos.head(5)
